In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Embedding model evaluation

This notebook compares three dense retrieval models on the same stratified Enron sample and the same 250 evaluation questions. Each model uses one fixed, model-appropriate chunking configuration.
- `sentence-transformers/all-MiniLM-L6-v2`: 200-token chunks with 50-token overlap.
- `BAAI/bge-base-en-v1.5`: 400-token chunks with 80-token overlap and the recommended retrieval query instruction.
- `intfloat/e5-base-v2`: 400-token chunks with 80-token overlap and the required `query:` and `passage:` prefixes.

The larger models support up to 512 tokens. A 400-token body budget leaves room for prefixes, special tokens, and small tokenization changes. Results are measured at the email level using Recall@1, Recall@3, Recall@5, Recall@10, and mean reciprocal rank (MRR).

In [ ]:
# Load the sampled Enron corpus and evaluation questions used to compare embedding models.
import gc
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

SAMPLED_EMAILS_FILE = Path("enron_stratified_sample.csv")
EVALUATION_QUESTIONS_FILE = Path("data/enron_evaluation_250.csv")
sampled_emails = pd.read_csv(SAMPLED_EMAILS_FILE)
evaluation_questions = pd.read_csv(EVALUATION_QUESTIONS_FILE)

print(f"Loaded {len(sampled_emails):,} sampled emails.")
print(f"Loaded {len(evaluation_questions):,} evaluation questions.")

In [ ]:
# Validate the evaluation data and confirm that every labelled source email is present.
REQUIRED_EMAIL_COLUMNS = {
    "email_id", "sender", "recipients", "subject", "body"
}
REQUIRED_EVALUATION_COLUMNS = {
    "question", "answer", "email_id", "difficulty", "retrieval_type"
}

missing_email_columns = REQUIRED_EMAIL_COLUMNS - set(sampled_emails.columns)
missing_evaluation_columns = (
    REQUIRED_EVALUATION_COLUMNS - set(evaluation_questions.columns)
)
assert not missing_email_columns, f"Missing email columns: {missing_email_columns}"
assert not missing_evaluation_columns, (
    f"Missing evaluation columns: {missing_evaluation_columns}"
)

sampled_emails["email_id"] = sampled_emails["email_id"].astype(str)
evaluation_questions["email_id"] = evaluation_questions["email_id"].astype(str)

missing_source_emails = evaluation_questions.loc[
    ~evaluation_questions["email_id"].isin(sampled_emails["email_id"])
]
assert missing_source_emails.empty, (
    f"{len(missing_source_emails)} labelled source emails are absent from the sample."
)
print("All labelled source emails are present in the sampled corpus.")

In [ ]:
# Combine email metadata and body text into the common representation used for retrieval.
def build_search_text(email_frame):
    """Build the canonical retrieval text used by every evaluation notebook."""
    return (
        email_frame["sender"].fillna("") + " " +
        email_frame["recipients"].fillna("") + " " +
        email_frame["subject"].fillna("") + " " +
        email_frame["body"].fillna("")
    )


sampled_emails["search_text"] = build_search_text(sampled_emails)

print("Search text prepared for every sampled email.")

## Model-specific settings

The BGE query instruction and E5 query/passage prefixes follow their model-card retrieval guidance. Document and question embeddings are normalized, making dot product equivalent to cosine similarity. Batch sizes are conservative defaults for the larger 768-dimensional models and can be reduced if the available device runs out of memory.

In [ ]:
# Define the embedding models and model specific retrieval settings being compared.
MODEL_CONFIGURATIONS = [
    {
        "model_label": "MiniLM-L6-v2",
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "chunk_size": 200,
        "chunk_overlap": 50,
        "query_prefix": "",
        "passage_prefix": "",
        "batch_size": 256,
    },
    {
        "model_label": "BGE-base-en-v1.5",
        "model_name": "BAAI/bge-base-en-v1.5",
        "chunk_size": 400,
        "chunk_overlap": 80,
        "query_prefix": (
            "Represent this sentence for searching relevant passages: "
        ),
        "passage_prefix": "",
        "batch_size": 64,
    },
    {
        "model_label": "E5-base-v2",
        "model_name": "intfloat/e5-base-v2",
        "chunk_size": 400,
        "chunk_overlap": 80,
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
        "batch_size": 64,
    },
]

display(pd.DataFrame(MODEL_CONFIGURATIONS))

In [ ]:
# Split emails into overlapping token chunks using each embedding model's tokenizer.
def create_model_chunks(embedding_model, configuration):
    """Create token-based email chunks using one model's tokenizer."""
    chunk_size = configuration["chunk_size"]
    chunk_overlap = configuration["chunk_overlap"]
    passage_prefix = configuration["passage_prefix"]

    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")
    if chunk_size >= embedding_model.max_seq_length:
        raise ValueError(
            "chunk_size must leave room for prefixes and special tokens"
        )

    tokenizer = embedding_model.tokenizer
    step = chunk_size - chunk_overlap
    chunk_records = []

    for email_record in sampled_emails.itertuples(index=False):
        token_ids = tokenizer.encode(
            str(email_record.search_text),
            add_special_tokens=False,
            truncation=False,
        )

        for chunk_number, start in enumerate(range(0, len(token_ids), step)):
            end = min(start + chunk_size, len(token_ids))
            chunk_token_ids = token_ids[start:end]
            if not chunk_token_ids:
                continue

            chunk_text = tokenizer.decode(
                chunk_token_ids,
                skip_special_tokens=True,
            )
            chunk_records.append({
                "email_id": email_record.email_id,
                "chunk_number": chunk_number,
                "chunk_id": (
                    f"{email_record.email_id}_chunk_{chunk_number}"
                ),
                "chunk_text": chunk_text,
                "embedding_text": passage_prefix + chunk_text,
                "token_count": len(chunk_token_ids),
            })

            # Prevent a redundant trailing chunk already covered by overlap.
            if end == len(token_ids):
                break

    return pd.DataFrame(chunk_records)

In [ ]:
# Build and evaluate one embedding model using email level retrieval Recall and MRR.

# Create a temporary chunk index for one model and measure source email retrieval performance.
def evaluate_embedding_model(configuration):
    """Build one temporary index and evaluate email-level retrieval."""
    model_started = time.perf_counter()
    embedding_model = SentenceTransformer(configuration["model_name"])

    print("\n" + "=" * 80)
    print("Model:", configuration["model_name"])
    print("Device:", embedding_model.device)
    print("Maximum sequence length:", embedding_model.max_seq_length)
    print(
        "Chunking:",
        configuration["chunk_size"],
        "tokens with",
        configuration["chunk_overlap"],
        "tokens of overlap",
    )

    chunk_started = time.perf_counter()
    model_chunks = create_model_chunks(embedding_model, configuration)
    chunking_seconds = time.perf_counter() - chunk_started
    print(f"Created {len(model_chunks):,} chunks.")

    embedding_started = time.perf_counter()
    chunk_embeddings = embedding_model.encode(
        model_chunks["embedding_text"].tolist(),
        batch_size=configuration["batch_size"],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

    prefixed_questions = [
        configuration["query_prefix"] + str(question)
        for question in evaluation_questions["question"]
    ]
    question_embeddings = embedding_model.encode(
        prefixed_questions,
        batch_size=configuration["batch_size"],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    embedding_seconds = time.perf_counter() - embedding_started

    retrieval_started = time.perf_counter()
    similarities = question_embeddings @ chunk_embeddings.T
    chunk_email_ids = model_chunks["email_id"].to_numpy()
    query_rank_records = []

    for question_index, scores in enumerate(similarities):
        relevant_email_id = evaluation_questions.iloc[question_index]["email_id"]
        ranked_chunk_indices = np.argsort(scores)[::-1]
        seen_email_ids = set()
        relevant_rank = None

        for chunk_index in ranked_chunk_indices:
            candidate_email_id = str(chunk_email_ids[chunk_index])
            if candidate_email_id in seen_email_ids:
                continue

            seen_email_ids.add(candidate_email_id)
            if candidate_email_id == relevant_email_id:
                relevant_rank = len(seen_email_ids)
                break

        query_rank_records.append({
            "model_label": configuration["model_label"],
            "model_name": configuration["model_name"],
            "question_index": question_index,
            "question": evaluation_questions.iloc[question_index]["question"],
            "email_id": relevant_email_id,
            "difficulty": evaluation_questions.iloc[question_index]["difficulty"],
            "retrieval_type": evaluation_questions.iloc[question_index]["retrieval_type"],
            "relevant_email_rank": relevant_rank,
        })

    retrieval_seconds = time.perf_counter() - retrieval_started
    query_ranks = pd.DataFrame(query_rank_records)
    ranks = query_ranks["relevant_email_rank"].fillna(np.inf).to_numpy()

    model_result = {
        "model_label": configuration["model_label"],
        "model_name": configuration["model_name"],
        "device": str(embedding_model.device),
        "model_max_sequence_length": embedding_model.max_seq_length,
        "embedding_dimensions": int(chunk_embeddings.shape[1]),
        "chunk_size": configuration["chunk_size"],
        "chunk_overlap": configuration["chunk_overlap"],
        "number_of_chunks": len(model_chunks),
        "recall_at_1": float(np.mean(ranks <= 1)),
        "recall_at_3": float(np.mean(ranks <= 3)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "recall_at_10": float(np.mean(ranks <= 10)),
        "mrr": float(np.mean(1.0 / ranks)),
        "chunking_seconds": chunking_seconds,
        "embedding_seconds": embedding_seconds,
        "retrieval_seconds": retrieval_seconds,
        "total_seconds": time.perf_counter() - model_started,
    }

    print(
        f"Recall@1={model_result['recall_at_1']:.1%} | "
        f"Recall@5={model_result['recall_at_5']:.1%} | "
        f"Recall@10={model_result['recall_at_10']:.1%} | "
        f"MRR={model_result['mrr']:.3f}"
    )

    del embedding_model
    del model_chunks
    del chunk_embeddings
    del question_embeddings
    del similarities
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return model_result, query_ranks

In [ ]:
# Evaluate each embedding model and collect overall and question type retrieval results.
RESULT_COLUMNS = [
    "model_label", "model_name", "device",
    "model_max_sequence_length", "embedding_dimensions",
    "chunk_size", "chunk_overlap", "number_of_chunks",
    "recall_at_1", "recall_at_3", "recall_at_5",
    "recall_at_10", "mrr", "chunking_seconds",
    "embedding_seconds", "retrieval_seconds", "total_seconds",
]

embedding_results = pd.DataFrame(columns=RESULT_COLUMNS)
all_query_ranks = pd.DataFrame()

for configuration in MODEL_CONFIGURATIONS:
    model_result, model_query_ranks = evaluate_embedding_model(configuration)
    embedding_results = pd.concat(
        [embedding_results, pd.DataFrame([model_result])],
        ignore_index=True,
    )
    all_query_ranks = pd.concat(
        [all_query_ranks, model_query_ranks],
        ignore_index=True,
    )


In [ ]:
# Rank the embedding models by retrieval performance and report the strongest result.
embedding_results = embedding_results[RESULT_COLUMNS].sort_values(
    ["recall_at_10", "recall_at_5", "mrr"],
    ascending=False,
).reset_index(drop=True)

display(
    embedding_results.style
    .format({
        "recall_at_1": "{:.1%}",
        "recall_at_3": "{:.1%}",
        "recall_at_5": "{:.1%}",
        "recall_at_10": "{:.1%}",
        "mrr": "{:.3f}",
        "chunking_seconds": "{:.1f}",
        "embedding_seconds": "{:.1f}",
        "retrieval_seconds": "{:.1f}",
        "total_seconds": "{:.1f}",
    })
    .background_gradient(
        subset=["recall_at_1", "recall_at_3", "recall_at_5", "recall_at_10", "mrr"],
        cmap="YlGn",
    )
)

best_model = embedding_results.iloc[0]
print(
    f"Best model by Recall@10: {best_model['model_name']} "
    f"({best_model['recall_at_10']:.1%})"
)

In [ ]:
# Compare model performance across question difficulty and retrieval type.
def recall_at_cutoff(ranks, cutoff):
    return (ranks.fillna(np.inf) <= cutoff).mean()

breakdown_results = (
    all_query_ranks
    .groupby(["model_label", "difficulty", "retrieval_type"])
    .agg(
        questions=("question_index", "count"),
        recall_at_1=("relevant_email_rank", lambda ranks: recall_at_cutoff(ranks, 1)),
        recall_at_5=("relevant_email_rank", lambda ranks: recall_at_cutoff(ranks, 5)),
        recall_at_10=("relevant_email_rank", lambda ranks: recall_at_cutoff(ranks, 10)),
    )
    .reset_index()
)

display(
    breakdown_results.style.format({
        "recall_at_1": "{:.1%}",
        "recall_at_5": "{:.1%}",
        "recall_at_10": "{:.1%}",
    })
)

In [ ]:
# Visualise Recall@K across the evaluated embedding models.
plot_data = embedding_results.melt(
    id_vars=["model_label"],
    value_vars=["recall_at_1", "recall_at_3", "recall_at_5", "recall_at_10"],
    var_name="metric",
    value_name="recall",
)

plt.figure(figsize=(11, 5))
sns.barplot(data=plot_data, x="metric", y="recall", hue="model_label")
plt.ylim(0, 1)
plt.xlabel("Retrieval metric")
plt.ylabel("Recall")
plt.title("Embedding model retrieval comparison")
plt.legend(title="Embedding model")
plt.show()